In [1]:
class ExperimentData:
    def __init__(self, df):
        self.df = df.copy()
        self.predictions = {}
        self.simulations = {} 
        
def train_val_test_split(weekly):
    unique_weeks = sorted(weekly['wm_yr_wk'].unique())
    
    test_weeks = unique_weeks[-13:]
    val_weeks = unique_weeks[-26:-13]
    train_weeks = unique_weeks[:-26]

    return (
        weekly[weekly['wm_yr_wk'].isin(train_weeks)].copy(),
        weekly[weekly['wm_yr_wk'].isin(val_weeks)].copy(),
        weekly[weekly['wm_yr_wk'].isin(test_weeks)].copy()
    )
    


def train_lgb(train_df, val_df, features):

    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    )

    model.fit(
        train_df[features],
        train_df['D_true'],
        eval_set=[(val_df[features], val_df['D_true'])],
        eval_metric='mse',
        callbacks=[lgb.early_stopping(25)]
    )

    return model

def build_segments(df):
    sku = df.groupby('sku_id').apply(lambda x: pd.Series({
        'mean': x['D_true'].mean(),
        'std': x['D_true'].std(),
        'nonzero': (x['D_true'] > 0).sum(),
        'T': len(x)
    }))

    sku['ADI'] = sku['T'] / sku['nonzero']
    sku['CV2'] = (sku['std'] / sku['mean']) ** 2

    def label(r):
        if r['ADI'] < 1.32 and r['CV2'] < 0.49:
            return 'Smooth'
        if r['ADI'] >= 1.32 and r['CV2'] < 0.49:
            return 'Intermittent'
        if r['ADI'] < 1.32 and r['CV2'] >= 0.49:
            return 'Erratic'
        return 'Lumpy'

    sku['segment'] = sku.apply(label, axis=1)
    return sku[['segment']]

def predict_baseline(model, df, features):
    return np.maximum(model.predict(df[features]), 0)


def predict_segmented(train_df, test_df, features):

    seg = build_segments(train_df)

    # ✅ 正确 merge
    train_df = train_df.merge(seg, on='sku_id', how='left')
    test_df  = test_df.merge(seg, on='sku_id', how='left')

    preds = np.zeros(len(test_df))

    for s in ['Smooth', 'Erratic', 'Intermittent', 'Lumpy']:

        train_s = train_df[train_df['segment'] == s]
        test_s  = test_df[test_df['segment'] == s]

        if len(train_s) == 0 or len(test_s) == 0:
            continue

        if s in ['Smooth', 'Erratic']:
            m = LGBMRegressor(n_estimators=300, learning_rate=0.05)
            m.fit(train_s[features], train_s['D_true'])
            preds[test_s.index] = m.predict(test_s[features])

        else:
            # Lumpy / Intermittent
            mean_map = train_s.groupby('sku_id')['D_true'].mean()
            preds[test_s.index] = test_s['sku_id'].map(mean_map).fillna(0)

    return np.maximum(preds, 0)

def simulate_inventory(df, q_col):
    results = []

    for sku, g in df.groupby('sku_id'):
        g = g.sort_values('week_order')

        I = 0

        for _, r in g.iterrows():
            Q = r[q_col]
            D = r['D_true']

            I_new = I + Q - D

            results.append({
                "sku_id": sku,
                "week_order": r['week_order'],
                "holding": r['c_h'] * max(I_new, 0),
                "shortage": r['c_u'] * max(-I_new, 0),
                "order": r['c_fix'] if Q > 0 else 0,
                "fulfilled": max(0.0, min(D, I + Q)),
                "total_cost": r['c_h'] * max(I_new, 0)
                              + r['c_u'] * max(-I_new, 0)
                              + (r['c_fix'] if Q > 0 else 0)
            })

            I = I_new

    return pd.DataFrame(results)

from scipy.stats import norm

def compute_Q(df, pred_col, std_col):
    cr = df['c_u'] / (df['c_u'] + df['c_h'])
    z = norm.ppf(cr.clip(1e-6, 1-1e-6))

    Q = df[pred_col] + z * df[std_col]
    return np.ceil(np.maximum(Q, 0))

def run_experiment(train_df, val_df, test_df, features):

    # ===== 1. model =====
    model = train_lgb(train_df, val_df, features)

    test_df = test_df.copy()

    # ===== 2. prediction =====
    test_df['D_pred_base'] = predict_baseline(model, test_df, features)
    test_df['D_pred_seg']  = predict_segmented(train_df, test_df, features)

    # ===== 3. std（无泄漏版本）✅
    std_map = train_df.groupby('sku_id')['D_true'].std()
    test_df['std_hist'] = test_df['sku_id'].map(std_map).fillna(1)

    # ===== 4. Q =====
    test_df['Q_base'] = compute_Q(test_df, 'D_pred_base', 'std_hist')
    test_df['Q_seg']  = compute_Q(test_df, 'D_pred_seg', 'std_hist')

    # ===== 5. simulate =====
    sim_base = simulate_inventory(test_df, 'Q_base')
    sim_seg  = simulate_inventory(test_df, 'Q_seg')

    # ===== 6. summary =====
    def summarize(sim):
        return {
            "cost": sim['total_cost'].sum(),
            "holding": sim['holding'].sum(),
            "shortage": sim['shortage'].sum(),
            "service": sim['fulfilled'].sum() / test_df['D_true'].sum()
        }

    return {
        "baseline": summarize(sim_base),
        "segmented": summarize(sim_seg)
    }

In [2]:
# =========================
# 0. 依赖
# =========================
import os
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from lightgbm import LGBMRegressor
# =========================
# 1. 你的 feature 列（你需要自己保证已存在）
# =========================
feature_cols = [
    'lag_1', 'lag_2', 'lag_3', 'lag_4',
    'sell_price', 'price_chg',
    'holiday_week', 'snap_week',
    'dept_id_enc', 'cat_id_enc', 'state_id_enc',
    'week_order'
]


# =========================
# 2. split data
# =========================


# 1. Load data
calendar = pd.read_csv('calendar.csv', parse_dates=['date'])
sell_price = pd.read_csv('sell_prices.csv')
sales = pd.read_csv('sales_train_evaluation.csv')

# 2. Weekly demand aggregation, respecting calendar weeks and holiday/snaps
calendar['holiday_week'] = calendar[['event_name_1', 'event_name_2']].notna().any(axis=1).astype(int)
calendar['snap_week'] = calendar[['snap_CA', 'snap_TX', 'snap_WI']].max(axis=1)
calendar = calendar.reset_index().rename(columns={'index': 'day_index'})
calendar['d'] = 'd_' + (calendar['day_index'] + 1).astype(str)
week_info = calendar[['d', 'wm_yr_wk', 'holiday_week', 'snap_week']].copy()

id_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
d_cols = [c for c in sales.columns if c.startswith('d_')]

# 1. 创建 天(d) 到 周(wm_yr_wk) 的映射字典
d_to_wk = calendar.set_index('d')['wm_yr_wk'].to_dict()

# 2. 找出所有的销量列 d_1, d_2...
d_cols = [c for c in sales.columns if c.startswith('d_')]

# 3. 在宽表层面按周聚合 (核心优化)
# 我们把销量列按对应的周进行分组求和
sales_weekly_wide = sales[id_cols + d_cols].set_index(id_cols)
# 转置 -> 映射周 -> 分组求和 -> 再转置回来
sales_weekly_wide.columns = [d_to_wk[c] for c in sales_weekly_wide.columns]
sales_weekly_wide = sales_weekly_wide.T.groupby(level=0).sum().T

# 4. 现在再执行 melt，此时的总行数只有：30490 SKU * 280 周 ≈ 850万行
weekly = sales_weekly_wide.reset_index().melt(
    id_vars=id_cols, 
    var_name='wm_yr_wk', 
    value_name='D_true'
)

# 5. 将 wm_yr_wk 转回 int 并合并日历特征 (holiday, snap)
weekly['wm_yr_wk'] = weekly['wm_yr_wk'].astype(int)

# 提取周级别的特征 (每周只要一个特征值即可)
week_features = calendar.groupby('wm_yr_wk').agg({
    'holiday_week': 'max',
    'snap_week': 'max'
}).reset_index()

weekly = weekly.merge(week_features, on='wm_yr_wk', how='left')
weekly = weekly.merge(
    sell_price[['item_id', 'store_id', 'wm_yr_wk', 'sell_price']],
    on=['item_id', 'store_id', 'wm_yr_wk'],
    how='left'
)
weekly = weekly.sort_values(['item_id', 'store_id', 'wm_yr_wk']).reset_index(drop=True)
weekly['week_order'] = weekly['wm_yr_wk'].rank(method='dense').astype(int)

# lag features
for lag in [1, 2, 3, 4]:
    weekly[f'lag_{lag}'] = weekly.groupby(['item_id', 'store_id'])['D_true'].shift(lag)

# price change
weekly['price_prev'] = weekly.groupby(['item_id', 'store_id'])['sell_price'].shift(1)
weekly['price_chg'] = (weekly['sell_price'] / weekly['price_prev'] - 1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
weekly['price_chg'] = weekly['price_chg'].astype(float)

# volatility
weekly['std_hist'] = (
    weekly
    .groupby(['item_id', 'store_id'])['D_true']
    .rolling(window=4, min_periods=2)
    .std()
    .reset_index(level=[0, 1], drop=True)
)
weekly['std_hist'] = weekly['std_hist'].fillna(weekly['D_true'].groupby([weekly['item_id'], weekly['store_id']]).transform('std'))
weekly['std_hist'] = weekly['std_hist'].fillna(weekly['D_true'].std()).clip(lower=0.01)

# c_u 缺货成本 c_h持有成本
# 假设利润率 ~= 35% 年库存成本率 = 20% /52转换到周 fixed ordering cost (固定订货成本)
weekly['c_u'] = weekly['sell_price'] * 0.35
weekly['c_h'] = weekly['sell_price'] * 0.20 / 52
weekly['c_fix'] = 5.0

for col in ['dept_id', 'cat_id', 'state_id']:
    weekly[col] = weekly[col].astype(str)
    weekly[f'{col}_enc'] = LabelEncoder().fit_transform(weekly[col])

weekly['sku_id'] = weekly['item_id'].astype(str) + '_' + weekly['store_id'].astype(str)
weekly = weekly.dropna(subset=['lag_1', 'lag_2', 'lag_3', 'lag_4', 'sell_price']).reset_index(drop=True)

unique_weeks = sorted(weekly['wm_yr_wk'].unique())
test_weeks = unique_weeks[-13:]
val_weeks = unique_weeks[-26:-13]
train_weeks = unique_weeks[:-26]

feature_cols = [
    'lag_1', 'lag_2', 'lag_3', 'lag_4',
    'sell_price', 'price_chg',
    'holiday_week', 'snap_week',
    'dept_id_enc', 'cat_id_enc', 'state_id_enc',
    'week_order'
]


train_df, val_df, test_df = train_val_test_split(weekly)

train_df = train_df.dropna(subset=feature_cols + ['D_true'])
val_df   = val_df.dropna(subset=feature_cols + ['D_true'])
test_df  = test_df.dropna(subset=feature_cols + ['D_true'])

results = run_experiment(train_df, val_df, test_df, feature_cols)

print("\n========== EXPERIMENT RESULT ==========")

for k, v in results.items():
    print(f"\n--- {k.upper()} ---")
    for m, val in v.items():
        if m == "service":
            print(f"{m}: {val:.2%}")
        else:
            print(f"{m}: {val:.2f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.076627 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1792
[LightGBM] [Info] Number of data points in the train set: 5878382, number of used features: 12
[LightGBM] [Info] Start training from score 10.033345
Training until validation scores don't improve for 25 rounds
Early stopping, best iteration is:
[68]	valid_0's l2: 66.7904
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027231 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1779
[LightGBM] [Info] Number of data points in the train set: 1858105, number of used features: 12
[LightGBM] [Info] Start training from score 18.332773
[LightGBM] [Info] Auto-choosing row-wise multi-threading, t